# CS781 Capstone Phase 1 Pretraining: Model 1: Deep LLaMA-Style Causal Decoder (~134.7M)
### Target Objective: BERTScore F1 Champion (Deep Hierarchical Legal Semantics)
* **Architecture:** LLAMA-Style Causal Decoder trained 100% from scratch
* **Parameters:** ~16 layers, d=768, H_q=12, H_kv=4, Tied Embeddings
* **Token Budget:** 145,000,000 tokens across legal corpus
* **Hardware:** NVIDIA T4 16GB (fp16 + SDPA, micro-batch 8, grad-accum 8)


## 1. Hugging Face Login
Authenticate to download dataset and upload checkpoints.


In [ ]:
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("WARNING: HF_TOKEN not found. Set it in Kaggle Secrets or environment.")


## 2. Check GPU & Hardware Acceleration
Verify NVIDIA T4 GPU and activate PyTorch 2.0 SDPA memory-efficient kernels.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    # Enable PyTorch SDPA memory-efficient attention (T4 hardware optimized)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False) # T4 does not support FA2


## 3. Configuration & Hyperparameters
All constants and hyperparameters configured specifically for this model.


In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil
import gc
from huggingface_hub import HfApi

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Experiment Metadata
EXPERIMENT_NAME = "model1-deep-llama-135m"
MODEL_DESCRIPTION = "Deep 16-layer LLaMA architecture with Grouped Query Attention (3:1) and 50% Supreme Court weighting to maximize deep contextual semantic representations for BERTScore F1."
TARGET_METRIC = "BERTScore F1 Champion (Deep Hierarchical Legal Semantics)"

# Hugging Face Repositories
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MASTER_REPO = "sarweshbarnwal/CS781-Legal-Models"
HF_TOKENIZER_REPO = "sarweshbarnwal/legal-tokenizer-32000"

# Dataset Proportions & Budget
MIXTURE_RATIOS = {
    "Phase_1_Pretraining/Train/train_supreme_court.json": 0.5,
    "Phase_1_Pretraining/Train/train_acts.json": 0.2,
    "Phase_1_Pretraining/Train/train_tribunals.json": 0.2,
    "Phase_1_Pretraining/Train/train_high_courts.json": 0.1
}
TRAIN_TOKENS = 145000000
VAL_TOKENS = 3_000_000
CONTEXT_LENGTH = 1024
VOCAB_SIZE = 32000
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Architecture Hyperparameters
NUM_LAYERS = 16
HIDDEN_SIZE = 768
NUM_HEADS = 12
NUM_KV_HEADS = 4
INTERMEDIATE_SIZE = 2304
HEAD_DIM = 64

# Training Optimization (Guaranteed No OOM on 16GB T4)
MICRO_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 0.0003
MIN_LEARNING_RATE = 3e-05
WARMUP_STEPS = 1000
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# Generation Controls (Leaderboard Score Maximization)
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.7
TOP_P = 0.90
REPETITION_PENALTY = 1.15

# Output Directories
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, EXPERIMENT_NAME)
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
print("Configuration loaded for:", EXPERIMENT_NAME)


## 4. Memory-Safe Remote Data Streamer
Streams legal texts directly from Hugging Face JSON files via HTTP Range. Zero multi-gigabyte downloads, zero disk or RAM crashes.


In [ ]:
from huggingface_hub import HfFileSystem
import json

fs = HfFileSystem(token=HF_TOKEN)

def stream_json_array_texts(repo_id, filename, max_docs=None):
    """
    Streams individual {"text": ...} records from remote JSON arrays over HTTP Range.
    Never loads 6+ GB or 10+ GB files into memory or disk.
    """
    with fs.open(f"datasets/{repo_id}/{filename}", "r", encoding="utf-8") as f:
        content = ""
        dec = json.JSONDecoder()
        doc_count = 0
        chunk_size = 262144 # 256 KB
        
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            content += chunk
            s = content.strip()
            if s.startswith('['):
                s = s[1:].lstrip()
            
            while s:
                try:
                    obj, end = dec.raw_decode(s)
                    if isinstance(obj, dict) and "text" in obj:
                        yield obj["text"]
                        doc_count += 1
                        if max_docs and doc_count >= max_docs:
                            return
                    s = s[end:].lstrip()
                    if s.startswith(','):
                        s = s[1:].lstrip()
                except json.JSONDecodeError:
                    break
            content = s

print("Streaming data helper ready.")


## 5. Train Custom 32K Legal Tokenizer 100% From Scratch
Zero external or pretrained tokenizers used. Streams a balanced sample of 20,000 legal documents to train 32K BPE in ~45s, saves it locally, and immediately pushes it to the Master HF Hub under `{EXPERIMENT_NAME}/tokenizer`.


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from transformers import PreTrainedTokenizerFast
from huggingface_hub import HfApi

print("Initializing Byte-Level BPE tokenizer model from scratch...")
raw_tokenizer = Tokenizer(BPE(unk_token="<unk>"))
raw_tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
raw_tokenizer.decoder = ByteLevelDecoder()

special_tokens = ["<bos>", "<eos>", "<unk>", "<pad>"]
trainer_bpe = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet()
)

# Stream 5,000 docs from each dataset (~20,000 docs total, ~30M words)
def tokenizer_stream_iterator():
    for rel_path in MIXTURE_RATIOS.keys():
        print(f"  -> Sampling 5,000 docs for tokenizer from: {os.path.basename(rel_path)} ...")
        for text in stream_json_array_texts(HF_DATASET_REPO, rel_path, max_docs=5000):
            yield text

print(f"Training 32,000-vocabulary BPE tokenizer from scratch...")
raw_tokenizer.train_from_iterator(tokenizer_stream_iterator(), trainer=trainer_bpe)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)
tokenizer.padding_side = "left"

print(f"Tokenizer trained from scratch successfully! Vocabulary Size: {len(tokenizer):,}")
print(f"Special Tokens: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}")

# Save tokenizer locally
tokenizer_dir = os.path.join(MODEL_OUTPUT_DIR, "tokenizer")
tokenizer.save_pretrained(tokenizer_dir)
print(f"Saved tokenizer locally to: {tokenizer_dir}")

# Push scratch-trained tokenizer to Master HF Hub under the specific model subfolder
try:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_MASTER_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=tokenizer_dir,
        path_in_repo=f"{EXPERIMENT_NAME}/tokenizer",
        repo_id=HF_MASTER_REPO,
        commit_message=f"Upload scratch-trained 32K legal tokenizer for {EXPERIMENT_NAME}"
    )
    print(f"Pushed custom tokenizer to Master Hub: {HF_MASTER_REPO}/{EXPERIMENT_NAME}/tokenizer")
except Exception as e:
    print(f"Hub upload notice: {e}")


## 6. Stream & Tokenize Datasets On-The-Fly (Peak RAM < 1.5 GB)
Streams each dataset, encodes directly to integer token IDs, and immediately frees memory. Zero raw texts stored in RAM.


In [ ]:
train_tokens = []
val_tokens = []

random.seed(SEED)

for rel_path, ratio in MIXTURE_RATIOS.items():
    quota_train = int(TRAIN_TOKENS * ratio)
    quota_val = int(VAL_TOKENS * ratio)
    print(f"\nStreaming & tokenizing {os.path.basename(rel_path)} (Quota: {quota_train:,} train tokens, {ratio*100:.0f}%)...")
    
    curr_train = []
    curr_val = []
    
    for text in stream_json_array_texts(HF_DATASET_REPO, rel_path):
        tokens = tokenizer.encode(text, add_special_tokens=False) + [tokenizer.eos_token_id]
        if len(curr_val) < quota_val:
            curr_val.extend(tokens)
        elif len(curr_train) < quota_train:
            curr_train.extend(tokens)
        else:
            break
            
    train_tokens.extend(curr_train[:quota_train])
    val_tokens.extend(curr_val[:quota_val])
    print(f"  -> Added {len(curr_train[:quota_train]):,} train tokens | {len(curr_val[:quota_val]):,} val tokens")
    
    del curr_train, curr_val
    gc.collect()

print(f"\nFinal Total Train Tokens in RAM: {len(train_tokens):,}")
print(f"Final Total Validation Tokens in RAM: {len(val_tokens):,}")


## 7. Create Contiguous 1024-Token Blocks
Slice continuous token stream into uniform 1024-token context sequences.


In [ ]:
def create_blocks(tokens, block_size):
    total_len = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_len]
    return [tokens[i:i + block_size] for i in range(0, total_len, block_size)]

train_blocks = create_blocks(train_tokens, CONTEXT_LENGTH)
val_blocks = create_blocks(val_tokens, CONTEXT_LENGTH)

del train_tokens, val_tokens
gc.collect()

print(f"Training sequences: {len(train_blocks):,} blocks of {CONTEXT_LENGTH}")
print(f"Validation sequences: {len(val_blocks):,} blocks of {CONTEXT_LENGTH}")


## 8. Build Hugging Face Dataset Objects
Convert token blocks to PyTorch format.


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})

train_dataset.set_format(type="torch", columns=["input_ids"])
val_dataset.set_format(type="torch", columns=["input_ids"])

del train_blocks, val_blocks
gc.collect()

print("Train Dataset:", train_dataset)
print("Val Dataset:", val_dataset)


## 9. Define LLAMA from Scratch & Apply Residual Scaling
Instantiate model with zero pretrained weights, Pre-RMSNorm, RoPE, SwiGLU, and Tied Embeddings.


In [ ]:
from transformers import LlamaConfig, LlamaForCausalLM

config = LlamaConfig(
    vocab_size=len(tokenizer),
    hidden_size=768,
    num_hidden_layers=16,
    num_attention_heads=12,
    num_key_value_heads=4,
    intermediate_size=2304,
    max_position_embeddings=CONTEXT_LENGTH,
    rms_norm_eps=1e-6,
    rope_theta=10000.0,
    hidden_act="silu",
    tie_word_embeddings=True,
    bos_token_id=tokenizer.bos_token_id or tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
)

model = LlamaForCausalLM(config)

# Residual Layer Scaling (1 / sqrt(2L)) for deep FP16 stability
def apply_residual_scaling(model, num_layers):
    scale = 1.0 / math.sqrt(2 * num_layers)
    with torch.no_grad():
        for name, p in model.named_parameters():
            if "o_proj.weight" in name or "down_proj.weight" in name:
                p.data.mul_(scale)

apply_residual_scaling(model, 16)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Architecture: LLAMA")
print(f"Total Parameters: {total_params / 1e6:.2f}M")
print(f"Trainable Parameters: {trainable_params / 1e6:.2f}M")
assert 50_000_000 <= total_params <= 300_000_000, f"Model size {total_params} out of competition bounds!"


## 10. Data Collator & Training Arguments with Continuous Hub Checkpoint Sync
Saves checkpoints every 500 steps, immediately pushes them to Master HF Hub, and keeps only 1 checkpoint locally to save Kaggle storage.


In [ ]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, TrainerCallback
from huggingface_hub import HfApi

class HubCheckpointSyncCallback(TrainerCallback):
    """
    Pushes each newly saved checkpoint directly to the Master HF Hub
    under {HF_MASTER_REPO}/{EXPERIMENT_NAME}/checkpoint-{step}.
    Keeps Kaggle local storage minimal while cloud backups remain permanent.
    """
    def __init__(self, repo_id, subfolder, token):
        self.api = HfApi(token=token)
        self.repo_id = repo_id
        self.subfolder = subfolder
        self.token = token

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        checkpoint_dir = os.path.join(args.output_dir, f"checkpoint-{step}")
        if os.path.exists(checkpoint_dir) and self.token:
            print(f"\n[HubSync] Pushing checkpoint-{step} to HF Hub ({self.repo_id}/{self.subfolder}/checkpoint-{step})...")
            try:
                self.api.upload_folder(
                    folder_path=checkpoint_dir,
                    path_in_repo=f"{self.subfolder}/checkpoint-{step}",
                    repo_id=self.repo_id,
                    commit_message=f"Auto-sync checkpoint-{step} [Step {step}]"
                )
                print(f"[HubSync] Checkpoint-{step} successfully backed up to HF Hub!")
            except Exception as e:
                print(f"[HubSync] Warning: Failed to sync checkpoint-{step} to Hub: {e}")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    per_device_eval_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    max_grad_norm=MAX_GRAD_NORM,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    save_steps=500,
    save_total_limit=1,
    seed=SEED,
    report_to="none"
)
print("Training arguments configured with active Hub Checkpoint Sync.")


## 11. Training with Full Crash Recovery Safeguard
Checks local disk first. If the Kaggle session restarted/crashed, it automatically pulls the latest checkpoint from Hugging Face Hub and resumes training without repeating any steps.


In [ ]:
from transformers import Trainer
from huggingface_hub import list_repo_files, snapshot_download
import glob

# Register Trainer with active Hub Checkpoint Sync
hub_callback = HubCheckpointSyncCallback(
    repo_id=HF_MASTER_REPO,
    subfolder=EXPERIMENT_NAME,
    token=HF_TOKEN
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[hub_callback]
)

def find_latest_checkpoint(local_dir, repo_id, subfolder, token):
    # 1. Check local disk first
    local_cps = glob.glob(os.path.join(local_dir, "checkpoint-*"))
    local_steps = []
    for cp in local_cps:
        part = os.path.basename(cp).replace("checkpoint-", "")
        if part.isdigit():
            local_steps.append((int(part), cp))
    if local_steps:
        best_step, best_path = max(local_steps, key=lambda x: x[0])
        print(f"[Resume] Found local checkpoint at step {best_step}: {best_path}")
        return best_path

    # 2. If session restarted/crashed (local disk wiped), recover from HF Hub!
    if token:
        print(f"[Resume] No local checkpoints found. Checking remote HF Hub ({repo_id}/{subfolder})...")
        try:
            files = list_repo_files(repo_id=repo_id, repo_type="model", token=token)
            prefix = f"{subfolder}/checkpoint-"
            hub_steps = []
            for f in files:
                if f.startswith(prefix):
                    rem = f[len(prefix):].split("/")[0]
                    if rem.isdigit():
                        hub_steps.append(int(rem))
            if hub_steps:
                latest_hub_step = max(hub_steps)
                print(f"[Resume] CRASH RECOVERY ACTIVATED: Found remote checkpoint-{latest_hub_step} on HF Hub!")
                print(f"[Resume] Downloading checkpoint-{latest_hub_step} from Hub to resume...")
                snapshot_download(
                    repo_id=repo_id,
                    repo_type="model",
                    allow_patterns=[f"{subfolder}/checkpoint-{latest_hub_step}/*"],
                    local_dir=OUTPUT_DIR,
                    token=token
                )
                recovered_path = os.path.join(local_dir, f"checkpoint-{latest_hub_step}")
                if os.path.exists(recovered_path):
                    print(f"[Resume] Successfully restored checkpoint-{latest_hub_step}! Resuming from step {latest_hub_step}...")
                    return recovered_path
        except Exception as e:
            print(f"[Resume] Remote checkpoint check notice: {e}")

    print("[Resume] No previous checkpoints found. Starting fresh training from step 0...")
    return None

last_checkpoint = find_latest_checkpoint(MODEL_OUTPUT_DIR, HF_MASTER_REPO, EXPERIMENT_NAME, HF_TOKEN)

start_time = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
training_time = time.time() - start_time

total_tokens_trained = len(train_dataset) * CONTEXT_LENGTH
tokens_per_sec = total_tokens_trained / max(training_time, 1)

print(f"Training completed in: {training_time / 3600:.2f} hours")
print(f"Effective throughput: {tokens_per_sec:,.0f} tokens/sec")


## 12. Evaluate Validation Perplexity
Calculates Cross-Entropy Loss and Perplexity (PPL = exp(loss)).


In [ ]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"Validation Loss: {eval_loss:.4f}")
print(f"Validation Perplexity: {perplexity:.2f}")


## 13. Plot Training and Validation Loss Curve


In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_steps, train_loss = [], []
eval_steps, eval_loss_vals = [], []

for log in logs:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_loss_vals.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss", alpha=0.8)
if eval_steps:
    plt.plot(eval_steps, eval_loss_vals, label="Validation Loss", marker='o')
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title(f"Pretraining Loss Curve - {EXPERIMENT_NAME}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:", LOSS_PLOT_PATH)


## 14. Save Final Model & Push to Master Hugging Face Hub
Uploads weights, tokenizer, and loss plot to your master repository.


In [ ]:
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print("Saved final model to:", MODEL_OUTPUT_DIR)

try:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_MASTER_REPO, repo_type="model", exist_ok=True)
    
    # Upload checkpoint folder
    api.upload_folder(
        folder_path=MODEL_OUTPUT_DIR,
        repo_id=HF_MASTER_REPO,
        path_in_repo=EXPERIMENT_NAME,
        commit_message=f"Upload {EXPERIMENT_NAME} weights and tokenizer"
    )
    # Upload loss plot
    if os.path.exists(LOSS_PLOT_PATH):
        api.upload_file(
            path_or_fileobj=LOSS_PLOT_PATH,
            path_in_repo=f"{EXPERIMENT_NAME}/loss_plot.png",
            repo_id=HF_MASTER_REPO
        )
    print(f"Successfully synced model artifacts to {HF_MASTER_REPO}/{EXPERIMENT_NAME}")
except Exception as e:
    print(f"Hub upload skipped or failed ({e}). Model is safely saved locally.")


## 15. Load Phase 1 Test Prompts from Hugging Face


In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data):,} test prompts.")


## 16. Load Final Model for High-Speed Inference


In [ ]:
generation_tokenizer = tokenizer
generation_model = LlamaForCausalLM.from_pretrained(MODEL_OUTPUT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()
generation_model.config.use_cache = True

print("Inference model ready on device:", device)


## 17. Metric-Optimized 128-Word Generation Function
Enforces Repetition Penalty 1.15, strict left-padding continuation slicing, and 128-word truncation.


In [ ]:
def generate_batch(prompts, model, tokenizer, device, target_words=128):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONTEXT_LENGTH - MAX_GENERATION_TOKENS
    ).to(device)

    # In left-padded batches, newly generated tokens strictly begin at index inputs['input_ids'].shape[1]
    prompt_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_GENERATION_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    predictions = []
    for i in range(len(prompts)):
        gen_tokens = outputs[i, prompt_len:]
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        gen_words = gen_text.split()[:target_words]
        predictions.append(" ".join(gen_words))

    return predictions


## 18. Generate Predictions for CodaBench Test Set


In [ ]:
predictions = {}
GEN_BATCH_SIZE = 16

items = list(test_data.items())
total_test = len(items)
print(f"Generating continuations for {total_test} test prompts...")

for start in range(0, total_test, GEN_BATCH_SIZE):
    batch = items[start:start + GEN_BATCH_SIZE]
    batch_ids = [str(k) for k, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    batch_preds = generate_batch(
        prompts=batch_prompts,
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS
    )

    for ex_id, pred_text in zip(batch_ids, batch_preds):
        predictions[ex_id] = {"text": pred_text}

    done = min(start + GEN_BATCH_SIZE, total_test)
    if done % 64 == 0 or done == total_test:
        print(f"Progress: {done}/{total_test} ({100*done/total_test:.1f}%)")

# Save prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"\nprediction.json successfully saved ({len(predictions)} entries): {PREDICTION_PATH}")


## 19. Create CodaBench submission.zip & Push to Hub
Compresses prediction.json into submission.zip and uploads to master repository.


In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Created submission zip:", SUBMISSION_ZIP)

# Verify zip
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    names = zipf.namelist()
    assert "prediction.json" in names, "prediction.json missing from zip!"
    print("Verification passed! ZIP contains:", names)

# Push submission zip to Master Hub
try:
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=SUBMISSION_ZIP,
        path_in_repo=f"{EXPERIMENT_NAME}/submission.zip",
        repo_id=HF_MASTER_REPO
    )
    api.upload_file(
        path_or_fileobj=PREDICTION_PATH,
        path_in_repo=f"{EXPERIMENT_NAME}/prediction.json",
        repo_id=HF_MASTER_REPO
    )
    print(f"\nSynced submission.zip and prediction.json to {HF_MASTER_REPO}/{EXPERIMENT_NAME}/")
except Exception as e:
    print(f"Hub upload skipped ({e}). submission.zip is ready locally in {SUBMISSION_ZIP}.")
